# 小さい言語モデル：文章から数値、確率、生成へ

11:45から12分。朝の初読後に使います。対象は固定したQwen3-0.6B。ColabはT4（16 GB）、MacはMPSを**自分で指定**します。初回ダウンロード時間は演習時間に含めません。
API料金は発生しません。Colabの利用料金・割当ては別です。既定ではモデル取得・実行・保存を行いません。

**最終スライドとの対応**：右上の番号20「実習：Notebook 03で、文章から候補確率までを見る」。11–19の仕組み説明と対応させて観察します。
小型言語モデル（SLM）も、文章をトークンへ分け、文脈から次の候補を予測します。今日は重みの再学習をせず、公開された重みを読み込んで観察します。


**この1ファイルで実行できます。** 他のNotebook・教材コード・授業設定の取得は不要です。上から順に実行します。準備用の定義セルは折りたたんであります。開くと普通のPythonとしてすべて読めます。
必要なもの：PyTorch・Transformersと固定版のモデル重み（初回はネット接続）。他のNotebookは開かなくて構いません。


## 始める前に：必要なファイルと学生の作業

**このNotebook（`03_open_weight_lab.ipynb`）以外に、教員から受け取る設定ファイルやAPIキーはありません。** 01・02の `connection.local.yaml` は使いません。モデルのファイルは、下の読込みセルがHugging Faceから固定版を取得します。自分でファイルを探したり、アップロードしたりする必要はありません。

1. 教材READMEのColabリンクからこのNotebookを開く。「ランタイム」→「ランタイムのタイプを変更」でT4 GPUを選ぶ。
2. 冒頭の準備・関数定義セルを上から実行し、「1. モデル用の2ライブラリ」で初回のみ `INSTALL = True` にして実行する。
3. 「2. 計算機と固定モデル」で `DEVICE = 'cuda'`、初回は `ALLOW_DOWNLOAD = True`、`LOAD_MODEL = True` にして実行する。**初回取得にはネット接続が必要**。モデル名・装置・パラメータ数が表示されたら準備完了。
4. 「3」で `TEXT` を記入し、`OBSERVE = True`。続く「4」「5」を順に実行し、トークン・数値・確率を観察する。
5. 質問する場合は「6」で `PROMPT` を記入して `PREVIEW = True` →「7」で `RUN = True` →「8」で自分の説明を記入して `SAVE = True`。
6. `保存:` に表示されたファイルを、Colab左側のフォルダアイコン（ファイル）からダウンロードする。通常は `.json`、論文の活動を指定した生成では `.jsonl` と `.md` も保存される。

観察用の文・質問は自分で入力します。論文の抜粋を使う場合は、教員が案内した掲載版を手元で開き、入力条件を確認してから必要な文章だけを `source_excerpt` へ貼り付けます。このモデルにPDFや画像を添付する操作はありません。T4が選べない・割り当たらない場合は教員・TAへ伝えてください。


## 入力例 → モデルの仕組みを観察する

**操作の順番**：初回のみ `INSTALL = True` → ColabはT4を選び `DEVICE = 'cuda'` → 初回取得時 `ALLOW_DOWNLOAD = True`、`LOAD_MODEL = True` → 観察セルで `OBSERVE = True`。Macでは自分で `DEVICE = 'mps'`、CPUなら `DEVICE = 'cpu'` を指定します。GPUを使えない場合に自動切替はしません。

### 文字列を数値にする例

```python
TEXT = '京都の名物は'
OBSERVE = True
```

**固定版Qwen3-0.6Bの実測例（2026-10-02、Mac MPS、教材の固定版）**：

```text
トークン数: 5
115806 '京都'
15767 'の'
13072 '名'
52853 '物'
15322 'は'
元の文へ戻す: 京都の名物は
形（トークン数, 次元）: (5, 1024)
```

**確認すること**：「京都」は1個、「名物」は2個に分かれます。1トークン＝1文字とも1単語とも限りません。埋め込みは各トークンに1024個の数値が対応し、画面には先頭4個だけ表示します。その4個を人間の意味ラベルとして読めるわけではありません。

### 次に来るトークンの例

続く確率セルは、候補の文字列と確率を5行表示します。同条件のMac実測では、先頭は `、`（約0.171）、次は `あります`（約0.109）でした。全候補の合計は約1、表示する上位5候補だけの合計は約0.450でした。

これは文章の続きを予測する確率で、回答の正答確率ではありません。装置・数値精度で値は変わり得ます。文を `京都で食べたいものは` に変えた場合はトークン化から実行し、候補が変わるかを見ます。

### 質問して短く生成する例

```python
PROMPT = '机にペンが2本あります。3本追加すると何本ですか。数字だけで答えてください。'
source_excerpt = ''
source_location = ''
activity_id = None  # 自作の観察。共通論文の記録にはしない
PREVIEW = True
```

プレビューに自分の質問と会話用の区切りが表示されたら、生成セルで `RUN = True`。**内容と形式の確認の目安**は `5` です。これは正答の目安であり、モデルが必ずそう出力するという保証ではありません。誤答や「数字だけ」を守らない回答もそのまま記録します。

`終了状態: completed` は生成が終了したこと、`output_limit` は出力上限に達したことです。どちらも正答の判定ではありません。自分の説明を保存欄に書いて `SAVE = True` にすると `.json` ができます。論文の活動を明示して生成した場合は、JSONL・Markdownの読解記録も作られます。

**使う関数**：`load_model`（固定モデルを読む）、`encode_prompt`（実際の入力を作る）、`generate`（短い生成）、`save_result`（保存）。トークン化・埋め込み・確率の計算は観察セルに直接書いてあります。

**生成の実測（2026-10-02、Mac MPS、上の固定モデル・質問）**：生回答は「2本追加すると、3本になります。」でした。生成は終了しましたが、計算結果も「数字だけ」という指定も満たしませんでした。生回答は直さず、本人の説明に誤りと根拠を残します。

**スライド20の2文比較**：この配布版は1回に1文を処理します。`TEXT`を`京都の名物は`から`京都の観光名所は`へ書き換え、「3. 文をトークンに分ける」→「4」→「5」をもう一度実行します。二つの表示を見比べ、入力・トークン数・上位候補・気付きを手元へ記録します。観察だけの結果は生成用の`SAVE`には含まれません。


In [ ]:
# 0. 保存先と必要なライブラリ（教材コードの取得は不要）
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()  # このNotebookを実行しているフォルダ
print('準備完了。授業設定と関数は以下のセルに含まれています。')

### 準備用の関数を定義する

以下も上から順に実行します。この段階ではAPI送信・モデル取得・回答の生成は行いません。折りたたみを開くと、記録、入力、送信などの関数を読めます。授業中に変更する欄は、この後の番号付きの手順にあります。

In [ ]:
# @title 準備：このNotebookで使うライブラリ
from __future__ import annotations
from copy import deepcopy
from datetime import datetime
from datetime import timezone
from html import escape
from math import isfinite
from pathlib import Path
from time import monotonic
from typing import Any
from urllib.parse import urlparse
from uuid import uuid4
import json
import os
import re

In [ ]:
# @title 準備：読む論文・範囲・設問の設定
from __future__ import annotations
course = {'common_paper_id': 'P01',
 'day1_source_paper_id': 'P01',
 'common_reading': {'material_id': 'P01',
                    'source_version': 'Nature VoR 2026-09-16; publisher PDF retrieved 2026-09-30',
                    'assigned_scope': {'pdf_pages': [1, 2, 3],
                                       'printed_pages': [1, 2, 3],
                                       'text_sections': ['Abstract (p1)',
                                                         'Overview of Paper2Agent (p2 right column): paragraph '
                                                         'from "We implemented Paper2Agent..." through "...across '
                                                         'multiple papers simultaneously."'],
                                       'figures': ['Fig.1a', 'Fig.1b'],
                                       'captions': ['Fig.1 caption (p3)'],
                                       'scope_id': 'P01_COMMON_v2_FINAL_SLIDES'},
                    'question_set_id': 'Q_COMMON_3',
                    'scope_status': 'confirmed'},
 'homework_policy': {'status': 'confirmed',
                     'common_doi': '10.1038/s41586-026-11044-y',
                     'journal_titles': ['Nature', 'Science'],
                     'publication_status': 'published',
                     'article_type': 'research',
                     'open_access_required': True,
                     'different_from_common': True,
                     'selection_required': True,
                     'reviewer': 'student'},
 'activities': {'P0': {'material_choice': False, 'phase': 'R0', 'reading_ref': 'common_reading'},
                'P1': {'material_choice': False, 'phase': 'R1', 'reading_ref': 'common_reading'},
                'L1': {'material_choice': False, 'phase': None, 'reading_ref': 'common_reading'},
                'L2': {'material_choice': False, 'phase': None, 'reading_ref': 'common_reading'},
                'V1': {'material_choice': False, 'phase': None, 'material_id': 'V01'},
                'V2': {'material_choice': False, 'phase': None, 'material_id': 'V02'},
                'V3': {'material_choice': False, 'phase': None, 'material_id': 'V01'},
                'V4': {'material_choice': False, 'phase': None, 'material_id': 'V02'},
                'P2a': {'material_choice': False, 'phase': 'R2', 'reading_ref': 'common_reading'},
                'P2b': {'material_choice': False, 'phase': 'R3', 'reading_ref': 'common_reading'},
                'P3': {'material_choice': False, 'phase': 'R3', 'reading_ref': 'common_reading'},
                'D2T1': {'material_choice': False, 'material_id': 'T01', 'phase': 'transfer'},
                'D2T2': {'material_choice': False, 'material_id': 'T01', 'phase': 'transfer'},
                'D2W1': {'material_choice': False, 'material_id': None, 'phase': None},
                'D2W2': {'material_choice': False, 'material_id': None, 'phase': None},
                'HW1': {'material_choice': True, 'phase': 'homework'},
                'P01_REVIEW': {'material_choice': False, 'phase': 'homework', 'reading_ref': 'common_reading'},
                'TRANSFER': {'material_choice': False, 'material_id': 'P02', 'phase': 'transfer'},
                'METHODS': {'material_choice': False, 'material_id': 'M01', 'phase': None},
                'PRACTICE': {'material_choice': False, 'material_id': 'PRACTICE', 'phase': 'practice'}},
 'question_sets': {'Q_COMMON_3': ['この研究の主要な主張は何か。',
                                  'それを支える根拠は何か。指定した本文・図と対応付けて説明する。',
                                  'まだ理解できていない点、追加で確認したい点は何か。'],
                   'Q_METHODS_5': ['ツールを取り出す前に、元のtutorialを実行する目的は何か。',
                                   '自動テストは何を照合し、失敗が続いたツールをどう扱うか。',
                                   'AlphaGenomeの3条件の比較では、使える道具・情報がどう違うか。',
                                   '自由記述の回答は、誰がどのような基準で評価したか。',
                                   '論文の範囲外の質問を、どのように作り、何を評価したか。'],
                   'Q_TRANSFER_3': ['研究のどの工程を自動化し、人がどこに関与しているか。',
                                    '図1の処理の流れと評価結果を区別して説明できるか。',
                                    '今回の評価で示されたことと、まだ保証されていないことは何か。']}}

manifest = {'materials': [{'id': 'P01',
                'role': 'common_paper',
                'bibliography': {'authors': ['Jiacheng Miao',
                                             'Joe R. Davis',
                                             'Yaohui Zhang',
                                             'Jonathan K. Pritchard',
                                             'James Zou'],
                                 'title': 'Reimagining research papers as interactive and reliable AI agents',
                                 'journal': 'Nature',
                                 'year': 2026,
                                 'article_type': 'Article'},
                'doi': '10.1038/s41586-026-11044-y',
                'source_url': 'https://www.nature.com/articles/s41586-026-11044-y',
                'file_hash': '6727ef9ca72cfffb1e032a2f07c7b9a4a2b235154bdd779b135c3518433677bf',
                'adoption': 'adopted',
                'rights': {'view': 'confirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'},
                'open_access': True,
                'publication_status': 'published',
                'article_type': 'research',
                'license': {'id': 'CC-BY-NC-ND-4.0',
                            'url': 'https://creativecommons.org/licenses/by-nc-nd/4.0/',
                            'verified_from': 'publisher webpage Rights and permissions; PDF p9'}},
               {'id': 'V01',
                'role': 'vlm_case',
                'source_ref': 'P01',
                'assigned_scope': {'scope_id': 'P01_FIG2_v2',
                                   'pdf_pages': [3, 4, 5],
                                   'figures': ['Fig.2a', 'Fig.2b', 'Fig.2c', 'Fig.2d'],
                                   'captions': ['Fig.2 caption (p4)'],
                                   'text_sections': ['AlphaGenome agent: example and benchmark (p3 right column '
                                                     'to p5 opening)']}},
               {'id': 'V02',
                'role': 'vlm_case',
                'source_ref': 'P01',
                'assigned_scope': {'scope_id': 'P01_FIG12_CONTEXT_v1',
                                   'pdf_pages': [2, 3, 4, 5],
                                   'figures': ['Fig.1a', 'Fig.1b', 'Fig.2a', 'Fig.2b', 'Fig.2c', 'Fig.2d'],
                                   'captions': ['Fig.1 caption (p3)', 'Fig.2 caption (p4)'],
                                   'text_sections': ['Overview and MCP components and testing (p2)',
                                                     'AlphaGenome agent: example and benchmark (p3 right column '
                                                     'to p5 opening)']}},
               {'id': 'T01',
                'role': 'transfer_case',
                'bibliography': None,
                'doi': None,
                'source_url': None,
                'file_hash': None,
                'adoption': 'pending',
                'rights': {'view': 'unconfirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'},
                'source_version': None,
                'assigned_scope': None},
               {'id': 'P02',
                'role': 'transfer_paper',
                'bibliography': {'authors': ['Chris Lu', 'Cong Lu', 'Robert Tjarko Lange', 'et al.'],
                                 'title': 'Towards end-to-end automation of AI research',
                                 'journal': 'Nature',
                                 'year': 2026,
                                 'volume': 651,
                                 'pages': '914–919'},
                'doi': '10.1038/s41586-026-10265-5',
                'source_url': 'https://www.nature.com/articles/s41586-026-10265-5',
                'source_version': 'Nature Version of Record 2026-03-25',
                'adoption': 'adopted',
                'assigned_scope': {'text_sections': ['Abstract (PDF p1)',
                                                     'Generating manuscripts (PDF p2)',
                                                     'Human evaluation results (PDF p3–4)',
                                                     'Limitations (PDF p4–5)'],
                                   'figures': ['Fig.1a', 'Fig.1b', 'Fig.1c'],
                                   'pdf_pages': [1, 2, 3, 4, 5],
                                   'printed_pages': [914, 915, 916, 917, 918]},
                'question_set_id': 'Q_TRANSFER_3',
                'file_hash': 'a75e0d93447f400179136bf18d909df29e0c8ccaeba076a1dfb1beeef0e0e10d',
                'open_access': True,
                'publication_status': 'published',
                'article_type': 'research',
                'license': {'id': 'CC-BY-4.0', 'url': 'https://creativecommons.org/licenses/by/4.0/'},
                'rights': {'view': 'confirmed',
                           'classroom': 'unconfirmed',
                           'ai_input': 'unconfirmed',
                           'redistribution': 'unconfirmed',
                           'slide_reproduction': 'unconfirmed'}},
               {'id': 'M01',
                'role': 'methods_case',
                'source_ref': 'P01',
                'assigned_scope': {'text_sections': ['Methods'], 'scope_id': 'P01_METHODS_DRAFT'},
                'question_set_id': 'Q_METHODS_5'},
               {'id': 'PRACTICE',
                'role': 'practice',
                'adoption': 'adopted',
                'bibliography': {'title': 'Notebookの操作練習（自作例。原論文ではない）'},
                'source_version': 'notebook-practice-2026-10-02',
                'assigned_scope': {'text_sections': ['このNotebookの入力例']},
                'rights': {'ai_input': 'confirmed'}}]}

In [ ]:
# @title 資料・設定を確認する関数：PHASES, ACTIVITY_NAMES, ValidationError, resolve_material, validate_homework_material
from __future__ import annotations
PHASES = {"P0": "R0", "P1": "R1", "P2a": "R2", "P2b": "R3", "P3": "R3"}


ACTIVITY_NAMES = {
    'PRACTICE': 'Notebookの操作練習（自作例）',
    'P0': '最初の読解（AIなし）', 'P1': '最初のAI読解',
    'L1': '言語モデルの仕組みを観察', 'L2': '質問方法の比較',
    'V1': '図2を画像で読む', 'V2': '画像の大きさを比べる',
    'P2a': '同じ3問への再回答', 'P3': '相互説明後の記録',
    'METHODS': 'Methodsの追加課題', 'TRANSFER': 'The AI Scientistへの応用',
    'HW1': 'Homework：別の原著論文', 'P01_REVIEW': 'Paper2Agentの追加読解',
}


class ValidationError(ValueError):
    """送信・配布前に修正が必要な設定。"""


def resolve_material(course: dict, manifest: dict, material_id: str) -> dict:
    """図の活動IDから原論文の版・権利を解決する。別論文への置換や循環を拒否。"""
    items = {m['id']: m for m in manifest['materials']}
    if material_id not in items:
        raise ValidationError('素材が登録されていません。')
    m = deepcopy(items[material_id])
    source_id = m.get('source_ref', material_id)
    if source_id != material_id:
        if source_id != course.get('day1_source_paper_id') or source_id not in items or items[source_id].get('source_ref'):
            raise ValidationError('第1日の図はP01を直接参照してください。')
        forbidden = {'bibliography', 'doi', 'source_url', 'source_version', 'file_hash', 'rights', 'adoption',
                     'pdf_url', 'pdf_page_count', 'license', 'open_access', 'publication_status', 'article_type'}
        if forbidden & m.keys():
            raise ValidationError('図の出典・版・権利・採用はP01から継承し、上書きしないでください。')
        m = {**deepcopy(items[source_id]), **m}
    if source_id == course['common_paper_id']:
        m['source_version'] = course['common_reading']['source_version']
        if material_id == source_id:
            m.update(deepcopy(course['common_reading']))
    m['source_material_id'] = source_id
    return m


def validate_homework_material(course: dict, material: dict) -> None:
    """掲載版・対象誌・OAの本人確認を検査。教員採用や科学的正しさは認定しない。"""
    policy = course.get('homework_policy', {})
    if policy.get('status') != 'confirmed':
        raise ValidationError('Homeworkの対象誌条件は教員確認待ちです。')
    doi_value = material.get('doi')
    doi = doi_value.strip().lower() if isinstance(doi_value, str) else ''
    common_doi = policy.get('common_doi', '').lower()
    if not common_doi:
        raise ValidationError('共通論文のDOI照合設定が必要です。')
    if material.get('id') == course['common_paper_id'] or doi == common_doi:
        raise ValidationError('HW1はP01と別論文を選んでください。P01継続はP01_REVIEWで記録できます。')
    bib = material.get('bibliography')
    if not isinstance(bib, dict) or not bib.get('title') or not bib.get('authors'):
        raise ValidationError('題名・著者・掲載誌を構造化した書誌に記入してください。')
    journal = bib.get('journal')
    if journal not in policy.get('journal_titles', []):
        raise ValidationError('Homeworkの対象誌リストに含まれていません。')
    if not re.fullmatch(r'10\.\d{4,9}/\S+', doi):
        raise ValidationError('掲載論文のDOIが必要です。')
    url = urlparse(material.get('source_url', ''))
    if url.scheme != 'https' or url.hostname not in {'www.nature.com', 'www.science.org'} or url.username or url.password:
        raise ValidationError('出版社の掲載論文ページを確認してください。')
    publisher = {'Nature': 'www.nature.com', 'Science': 'www.science.org'}
    if journal in publisher and url.hostname != publisher[journal]:
        raise ValidationError('掲載誌と出版社ページが一致しません。')
    if material.get('publication_status') != 'published' or material.get('article_type') != 'research':
        raise ValidationError('掲載済みの原著論文を選んでください。プレプリント・会議・ニュース等は対象外です。')
    if material.get('open_access') is not True:
        raise ValidationError('Open Access表示と利用条件を確認してください。')
    review = material.get('eligibility_review', {})
    if review.get('status') != 'confirmed' or not review.get('reviewer'):
        raise ValidationError('対象誌・原著・版・OAの確認状態と確認者を記録してください。')

In [ ]:
# @title 資料・設定を確認する関数：material_context
from __future__ import annotations
def material_context(course: dict, manifest: dict, activity_id: str,
                     selection: str | None = None, local_material: dict | None = None,
                     for_input: bool = True) -> dict[str, Any]:
    """共通課題はP01固定。Homeworkの学生選定だけ別のローカル定義を使う。"""
    if activity_id not in course["activities"]:
        raise ValidationError("活動IDを確認してください。")
    a = course["activities"][activity_id]
    if a.get("reading_ref") == "common_reading":
        mid = course["common_paper_id"]
        if selection is not None and selection != mid:
            raise ValidationError("共通課題の素材はP01に固定されています。")
    elif a.get("material_choice"):
        mid = selection
        if not mid:
            raise ValidationError("Homeworkの素材を明示してください。")
    else:
        mid = a.get("material_id")
        if selection is not None and selection != mid:
            raise ValidationError("この活動の素材は固定されています。")
    if mid and mid.startswith("U_"):
        if not a.get("material_choice") or not re.fullmatch(r"U_[A-Za-z0-9_-]+", mid):
            raise ValidationError("学生選定資料はHomework等で使用します。")
        m = deepcopy(local_material or {})
        if m.get("id") != mid or not all(m.get(k) for k in ("source_url", "bibliography", "source_version", "assigned_scope")):
            raise ValidationError("学生選定資料の出典・書誌・版・範囲を記入してください。")
        if for_input and (m.get("input_review", {}).get("status") != "confirmed" or not m.get("input_review", {}).get("reviewer")):
            raise ValidationError("入力の可否と確認者を記録してください。不明なら送らず教員へ相談してください。")
        m.update(role="student_paper", selection_origin="student", adoption="not_applicable", publication="not_granted")
    else:
        m = resolve_material(course, manifest, mid)
        if a.get("material_choice") and m["role"] not in {"common_paper", "extension_paper"}:
            raise ValidationError("HW1は条件を満たす教員の追加論文・学生選定論文を使います。")
        if for_input and (m.get("adoption") != "adopted" or m.get("rights", {}).get("ai_input") != "confirmed"):
            raise ValidationError("素材の採用またはAI入力の確認待ちです。")
        m["selection_origin"] = "common" if mid == "P01" else "instructor"
    if activity_id == 'HW1' and for_input:
        validate_homework_material(course, m)
    m["material_id"] = mid
    m["phase"] = a.get("phase")
    m["activity_id"] = activity_id
    return m

In [ ]:
# @title 記録を作り、保存・再読込みする関数：FIELDS, new_record, validate_record, _unique_records, _json_object, _reject_json_constant, _finite_json_float, _read_json, record_json
from __future__ import annotations
FIELDS = """session_id run_id activity_id phase timestamp material_id material_role selection_origin
source_version file_hash assigned_scope input_scope page_or_figure service access_route requested_model
reported_model conversation_mode prior_turn_count tool_usage input_kind prompt visible_system_instructions
parameters_requested parameters_confirmed response_raw status error_type retry_count usage cost_value cost_kind
elapsed_seconds student_explanation evidence_location verification_status peer_insight unresolved_point next_change
student_revision change_reason question_set_id input_review input_provenance error_message request_input""".split()


def new_record(context: dict, session_id: str | None = None) -> dict[str, Any]:
    """空の個人記録を作る。費用不明や該当なしはnullを維持する。"""
    r = dict.fromkeys(FIELDS)
    r.update(session_id=session_id or str(uuid4()), run_id=str(uuid4()), timestamp=datetime.now(timezone.utc).isoformat(),
             status="draft", cost_kind="unknown", retry_count=0, visible_system_instructions=[])
    for key in ("activity_id", "phase", "material_id", "selection_origin", "source_version", "assigned_scope", "question_set_id", "input_review"):
        r[key] = deepcopy(context.get(key))
    r["material_role"] = context.get("role")
    # 追加項目として保存し、旧記録の読込み互換性を保つ。
    r['source_material_id'] = context.get('source_material_id', context.get('material_id'))
    r['source_file_hash'] = context.get('file_hash')
    for key in ('eligibility_review', 'bibliography', 'doi', 'source_url', 'file_hash',
                'publication_status', 'article_type', 'open_access', 'license'):
        r[key] = deepcopy(context.get(key))
    return r


def validate_record(record: dict) -> None:
    """P01の段階とHomeworkの混在、費用の誤表示を防ぐ。"""
    if not isinstance(record, dict) or any(k not in record for k in FIELDS):
        raise ValidationError("記録の必須フィールドが不足しています。")
    aid, phase = record["activity_id"], record["phase"]
    if aid in PHASES and phase != PHASES[aid]:
        raise ValidationError("共通課題とR0～R3の対応が不正です。")
    if phase in {"R0", "R1", "R2", "R3"} and (record["material_id"] != "P01" or PHASES.get(aid) != phase):
        raise ValidationError("R0～R3は対応するP01共通課題専用です。")
    if aid == "HW1" and phase != "homework":
        raise ValidationError("HW1はhomework記録として保存してください。")
    fixed = {'METHODS': ('M01', None, 'Q_METHODS_5'), 'TRANSFER': ('P02', 'transfer', 'Q_TRANSFER_3')}
    if aid in fixed and (record['material_id'], phase, record['question_set_id']) != fixed[aid]:
        raise ValidationError('Methodsと別論文への応用は、それぞれ固定した素材・段階・設問で保存してください。')
    if phase == "R0" and (record["response_raw"] is not None or record["service"] not in {None, "none"}):
        raise ValidationError("R0はAIなしの記録です。")
    if record["cost_kind"] == "unknown" and record["cost_value"] is not None:
        raise ValidationError("費用不明を数値で埋めないでください。")
    if not all(isinstance(record[k], str) and record[k].strip() for k in ("session_id", "run_id")):
        raise ValidationError("記録IDが必要です。")
    for key in ("cost_value", "elapsed_seconds"):
        value = record[key]
        if value is not None and (type(value) not in (int, float) or not isfinite(value) or value < 0):
            raise ValidationError("費用・所要時間は非負の有限数またはnullにしてください。")


def _unique_records(records: list[dict]) -> None:
    """保存と再読込みの両方で、記録の配列とrunの一意性を確認する。"""
    if not isinstance(records, list):
        raise ValidationError("記録は配列として保存してください。")
    for record in records:
        validate_record(record)
    ids = [(r["session_id"], r["run_id"]) for r in records]
    if len(ids) != len(set(ids)):
        raise ValidationError("同じrunが重複しています。")


def _json_object(pairs: list[tuple]) -> dict:
    """同名フィールドが二つあるJSONを、後勝ちで読まない。"""
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValidationError("JSONのフィールド名が重複しています。元の記録を確認してください。")
        value[key] = item
    return value


def _reject_json_constant(value: str) -> None:
    """NaNやInfinityはJSONの数値として受け付けない。"""
    raise ValidationError("JSONに非標準の数値が含まれています。")


def _finite_json_float(value: str) -> float:
    """桁あふれをInfinityへ変換せず拒否する。"""
    number = float(value)
    if not isfinite(number):
        raise ValidationError("JSONの数値が大きすぎます。")
    return number


def _read_json(content: str) -> Any:
    """壊れたJSONを補修・欠落扱いせず、明示的に拒否する。"""
    try:
        return json.loads(content, object_pairs_hook=_json_object, parse_constant=_reject_json_constant,
                          parse_float=_finite_json_float)
    except json.JSONDecodeError:
        raise ValidationError("JSONを読み込めません。元の記録ファイルを確認してください。") from None


def record_json(value: Any, *, indent: int | None = None) -> str:
    """保存で型や項目が変わる値を、ファイル作成前に拒否する。"""
    try:
        content = json.dumps(value, ensure_ascii=False, indent=indent, allow_nan=False)
    except (TypeError, ValueError):
        raise ValidationError("記録にJSONで保存できない値があります。文字列・数値・null・リスト・辞書を使ってください。") from None
    if _read_json(content) != value:
        raise ValidationError("保存すると記録の型が変わります。辞書のキーは文字列、配列はリストにしてください。")
    return content

In [ ]:
# @title 記録を作り、保存・再読込みする関数：_readable_response, _text_block, save_records
from __future__ import annotations
def _readable_response(raw: Any) -> str:
    """壊れた応答も保存できるよう、表示可能な文章だけを一覧へ取り出す。"""
    if isinstance(raw, str):
        return raw
    if raw is None:
        return 'AIの応答なし'
    texts = []
    if isinstance(raw, dict) and isinstance(raw.get('output'), list):
        for item in raw['output']:
            if not isinstance(item, dict) or not isinstance(item.get('content'), list):
                continue
            for part in item['content']:
                if isinstance(part, dict) and part.get('type') == 'output_text' and isinstance(part.get('text'), str):
                    texts.append(part['text'])
    return '\n'.join(texts) if texts else '文章として表示できる応答がありません。状態と下の全項目を確認してください。'


def _text_block(value: Any) -> str:
    """質問・回答にMarkdownやHTMLがあっても、記録の構造を変えず原文として表示する。"""
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, indent=2)
    return '<pre>' + escape(text) + '</pre>'


def save_records(records: list[dict], path: str | Path) -> None:
    """明示操作で新しいファイルへ保存。同名上書きで初期記録を失わない。"""
    _unique_records(records)
    p = Path(path)
    if p.suffix not in {".jsonl", ".md"}:
        raise ValidationError("保存形式は.mdまたは.jsonlです。")
    if p.suffix == ".jsonl":
        content = "".join(record_json(r) + "\n" for r in records)
    else:
        lines = ["# 個人の読解記録", "", "生出力と本人の説明を区別。共有前に内容を確認してください。", ""]
        for number, r in enumerate(records, 1):
            cost = ('外部APIなし' if r['cost_kind'] == 'not_applicable' else
                    str(r['cost_value']) + ' USD' if r['cost_value'] is not None else '不明（0 USDとは扱わない）')
            # 記録の内部IDと、学生が読む活動名を分ける。
            activity = ACTIVITY_NAMES.get(r['activity_id'], str(r['activity_id']))
            model = r['reported_model'] or r['requested_model'] or ('AIなし' if r['service'] in {None, 'none'} else '不明')
            status = {'manual': '本人が記入', 'success': '応答完了', 'failed': '失敗',
                      'incomplete': '未完了', 'draft': '記入途中', 'pending': '処理中'}.get(r['status'], str(r['status']))
            mode = {'new': '新規', 'continue': '履歴を継続', None: '該当なし'}.get(r['conversation_mode'], str(r['conversation_mode']))
            lines += [f"## 記録 {number}: {escape(activity)}", "",
                      f"- 状態: {escape(status)}",
                      f"- モデル: {escape(str(model))}",
                      f"- 会話: {escape(mode)} / 過去の対話 {r['prior_turn_count'] or 0} 回",
                      f"- 費用: {cost}", ""]
            bibliography = r.get('bibliography')
            if isinstance(bibliography, dict) and bibliography.get('title'):
                lines += ['資料: ' + escape(str(bibliography['title'])), '']
            if r.get('error_message'):
                lines += ['### エラー・未完了の理由', '', _text_block(r['error_message']), '']
            for field, label in [('prompt', '送った質問'), ('input_scope', '送った資料の範囲'),
                                 ('student_explanation', '本人の説明'), ('evidence_location', '根拠の場所'),
                                 ('student_revision', '修正した説明'), ('unresolved_point', '残る疑問'),
                                 ('next_change', '次に変えること'), ('change_reason', '変更した理由')]:
                if r.get(field) is not None:
                    lines += ['### ' + label, '', _text_block(r[field]), '']
            lines += ['### AIの生回答', '', _text_block(_readable_response(r['response_raw'])), '']
        lines += ["## 再読込み用の全項目", "", '<details>', '<summary>入力・履歴・添付を含む全項目（再読込み時はこの部分を変更しない）</summary>', '',
                  "<!-- SEMINAR_RECORDS -->", "```json", record_json(records, indent=2),
                  "```", "<!-- END_SEMINAR_RECORDS -->", '', '</details>', '']
        content = "\n".join(lines)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("x", encoding="utf-8") as f:
        f.write(content)

In [ ]:
# @title 入力の表示と保存先を作る関数：save_pair
from __future__ import annotations
def save_pair(records: list[dict], directory: str | Path = "outputs") -> tuple[Path, Path]:
    """手元へ保存する。サーバへ自動転送しない。"""
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
    paths = tuple(Path(directory) / (suffix + ext) for ext in (".jsonl", ".md"))
    for path in paths:
        save_records(records, path)
    return paths

In [ ]:
# @title 公開モデルの読込み・生成・保存の関数：MODEL_ID, REVISION, MAX_INPUT_TOKENS, MAX_NEW_TOKENS, configure_cache, load_model, encode_prompt, generate
from __future__ import annotations
MODEL_ID = 'Qwen/Qwen3-0.6B'


REVISION = 'c1899de289a04d12100db370d81485cdf75e47ca'


MAX_INPUT_TOKENS = 2048


MAX_NEW_TOKENS = 128


def configure_cache(root: Path) -> Path:
    """教材フォルダ内へキャッシュを固定する。HFの自動認証・telemetryを使わない。"""
    root = root.resolve()
    cache = root / 'build/open-weight-cache'
    if not cache.resolve().is_relative_to(root):
        raise ValueError('キャッシュの実体は教材フォルダ内に置いてください。')
    cache.mkdir(parents=True, exist_ok=True)
    os.environ.update(HF_HOME=str(cache / 'huggingface'), TORCH_HOME=str(cache / 'torch'),
                      XDG_CACHE_HOME=str(cache), HF_HUB_DISABLE_TELEMETRY='1',
                      HF_HUB_DISABLE_IMPLICIT_TOKEN='1', TOKENIZERS_PARALLELISM='false')
    return cache


def load_model(device: str, allow_download: bool = False):
    """指定した計算機だけで読み込む。失敗時の別モデル・CPUへの切替は行わない。"""
    if device not in {'cuda', 'mps', 'cpu'}:
        raise ValueError('deviceはcuda（Colab GPU）、mps（Mac）、cpuから明示してください。')
    if type(allow_download) is not bool:
        raise ValueError('allow_downloadはTrueまたはFalseで明示してください。')
    if os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK') == '1':
        raise ValueError('MPSからCPUへの自動フォールバックを無効にしてください。')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from huggingface_hub import snapshot_download
    if device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('GPUを利用できません。Colabのランタイムを確認してください。')
    if device == 'mps' and not torch.backends.mps.is_available():
        raise RuntimeError('この実行環境ではMac GPUを利用できません。')
    # local_files_onlyでもTransformersのモデル名判定が通信する版がある。
    # 固定snapshotを先に解決し、読込みにはローカルのパスだけを渡す。
    snapshot = snapshot_download(MODEL_ID, revision=REVISION, local_files_only=not allow_download,
        token=False, allow_patterns=['*.json', '*.safetensors', 'merges.txt', 'vocab.*', 'tokenizer.model'])
    options = dict(local_files_only=True, trust_remote_code=False, token=False)
    tokenizer = AutoTokenizer.from_pretrained(snapshot, **options)
    model = AutoModelForCausalLM.from_pretrained(
        snapshot, **options, use_safetensors=True,
        dtype=torch.float32 if device == 'cpu' else torch.float16,
        attn_implementation='eager').to(device).eval()
    return tokenizer, model


def encode_prompt(tokenizer, prompt: str):
    """画面で確認できる会話テンプレートを適用。質問の追加・履歴切捨てはしない。"""
    if not prompt.strip():
        raise ValueError('質問を自分で記入してください。')
    text = tokenizer.apply_chat_template([{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    encoded = tokenizer(text, return_tensors='pt', add_special_tokens=False)
    if encoded.input_ids.shape[1] > MAX_INPUT_TOKENS:
        raise ValueError(f'入力が{MAX_INPUT_TOKENS}トークンを超えています。自分で範囲を見直してください。')
    return text, encoded


def generate(tokenizer, model, prompt: str, max_new_tokens: int = 96, seed: int = 42) -> dict:
    """新規会話を1回実行。生回答・入力テンプレート・計算機・所要時間を保存可能にする。"""
    if type(max_new_tokens) is not int or not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f'出力上限は1〜{MAX_NEW_TOKENS}にしてください。')
    import torch
    from transformers import set_seed
    visible, encoded = encode_prompt(tokenizer, prompt)
    set_seed(seed)
    inputs = encoded.to(model.device)
    start = monotonic()
    with torch.inference_mode():
        result = model.generate(**inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.8, top_k=20,
            pad_token_id=tokenizer.eos_token_id)
    if model.device.type == 'mps':
        torch.mps.synchronize()
    elif model.device.type == 'cuda':
        torch.cuda.synchronize()
    ids = result[0, inputs.input_ids.shape[1]:].tolist()
    eos = model.generation_config.eos_token_id
    eos_ids = eos if isinstance(eos, list) else [eos]
    response = tokenizer.decode(ids, skip_special_tokens=True)
    status = _generation_status(ids, response, eos_ids, max_new_tokens)
    return {'run_id': str(uuid4()), 'timestamp': datetime.now(timezone.utc).isoformat(),
        'model': MODEL_ID, 'revision': REVISION, 'route': 'open_weight_local',
        'device': str(model.device), 'dtype': str(model.dtype), 'prompt': prompt,
        'visible_template': visible, 'conversation_mode': 'new', 'seed': seed,
        'parameters': {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20,
                       'max_new_tokens': max_new_tokens, 'enable_thinking': False},
        'input_tokens': int(inputs.input_ids.shape[1]), 'output_tokens': len(ids),
        'response_raw': response,
        'status': status,
        'elapsed_seconds': round(monotonic() - start, 3), 'api_cost_usd': 0,
        'student_explanation': None, 'unresolved_point': None}

In [ ]:
# @title 公開モデルの読込み・生成・保存の関数：_generation_status, reading_record, save_result
from __future__ import annotations
def _generation_status(ids: list[int], response: str, eos_ids: list[int], limit: int) -> str:
    """終了トークンか出力上限による終了だけを受け付ける。空の回答はエラー。"""
    if not ids or not response.strip():
        raise RuntimeError('モデルが空の回答を返しました。成功として保存せず、入力と出力上限を確認してください。')
    if len(ids) > limit:
        raise RuntimeError('指定した出力上限を超えています。モデルの設定を確認してください。')
    if ids[-1] in eos_ids:
        return 'completed'
    if len(ids) == limit:
        return 'output_limit'
    raise RuntimeError('想定した終了条件に達せず停止しました。モデルの設定を確認してください。')


def reading_record(result: dict, context: dict, source_location: str) -> dict:
    """公開重みの実行をAPIと同じ読解記録へ変換。生の結果と追加項目も保持する。"""
    from copy import deepcopy
    if result.get('model') != MODEL_ID or result.get('revision') != REVISION:
        raise ValueError('記録するモデルと固定版が一致しません。')
    if result.get('status') not in {'completed', 'output_limit'} or not result.get('response_raw'):
        raise ValueError('保存する生成結果がありません。')
    record = new_record(context)
    record.update(run_id=result['run_id'], timestamp=result['timestamp'],
        service='Transformers', access_route='open_weight_local', requested_model=MODEL_ID,
        reported_model=MODEL_ID, conversation_mode='new', prior_turn_count=0,
        prompt=result['prompt'], input_kind=['text'], input_scope=source_location,
        request_input=result['visible_template'], visible_system_instructions=[],
        parameters_requested=deepcopy(result['parameters']), parameters_confirmed=deepcopy(result['parameters']),
        response_raw=result['response_raw'], status=result['status'], elapsed_seconds=result['elapsed_seconds'],
        usage={'input_tokens': result['input_tokens'], 'output_tokens': result['output_tokens']},
        cost_kind='not_applicable', cost_value=None, open_weight_result=deepcopy(result))
    record['cost_note'] = '外部API呼出しなし。計算機・Colabの利用料金を推定していない。'
    validate_record(record)
    return record


def save_result(result: dict, annotations: dict, root: Path) -> tuple[dict, list[Path]]:
    """生の生成結果と本人の説明を一度で保存する。既存ファイルは上書きしない。"""
    from copy import deepcopy

    if (not isinstance(result, dict) or not isinstance(result.get('run_id'), str)
            or not result['run_id'].strip()):
        raise ValueError('保存する生成結果がありません。')
    if (result.get('model') != MODEL_ID or result.get('revision') != REVISION
            or result.get('route') != 'open_weight_local'):
        raise ValueError('このNotebookの固定モデルで生成した結果だけを保存してください。')
    if (result.get('status') not in {'completed', 'output_limit'}
            or not isinstance(result.get('response_raw'), str) or not result['response_raw'].strip()):
        raise ValueError('保存する生成結果がありません。終了状態と生回答を確認してください。')
    allowed = {'student_explanation', 'evidence_location', 'unresolved_point'}
    if (not isinstance(annotations, dict) or set(annotations) - allowed
            or any(v is not None and not isinstance(v, str) for v in annotations.values())):
        raise ValueError('記入欄は本人の説明・根拠・不明点の文字列かNoneにしてください。')
    saved = {**deepcopy(result), **{k: v for k, v in annotations.items() if v is not None}}
    record = None
    if saved.get('reading_context') is not None:
        record = reading_record(saved, saved['reading_context'], saved.get('source_location', ''))
        record.update({k: saved[k] for k in allowed if k in saved})
        validate_record(record)
    root = root.resolve()
    folder = root / 'outputs/open-weight'
    if not folder.resolve().is_relative_to(root):
        raise ValueError('保存先は教材フォルダ内にしてください。')
    destination = folder / (saved['run_id'] + '-' + uuid4().hex[:8] + '.json')
    if destination.resolve().parent != folder.resolve():
        raise ValueError('保存する実行番号に不正なパスが含まれています。')
    content = record_json(saved, indent=2)
    folder.mkdir(parents=True, exist_ok=True)
    with destination.open('x', encoding='utf-8') as handle:
        handle.write(content)
    paths = [destination]
    if record is not None:
        paths.extend(save_pair([record], root / 'outputs'))
    return saved, paths

## 1. モデル用の2ライブラリだけを準備する

In [ ]:
CACHE = configure_cache(ROOT)
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--cache-dir', str(CACHE / 'pip'),
                    'torch==2.8.0', 'transformers==4.57.6'], check=True)
print(MODEL_ID, REVISION)

## 2. 計算機と固定モデルを選ぶ
GPU未割当てはエラーです。CPUや別モデルに自動変更しません。
パラメータは学習で調整する数値です。`0.6B`は約6億を表しますが、名前から推測せず、読み込んだモデルの数値の個数を数えます。トークンの種類の数（語彙サイズ）とは別です。


In [ ]:
DEVICE = 'cuda'  # Colab: cuda / Mac: mps / CPU: cpu
ALLOW_DOWNLOAD = False
LOAD_MODEL = False
if LOAD_MODEL:
    tokenizer = model = None
    from importlib.metadata import version, PackageNotFoundError
    required_versions = {'torch': '2.8.0', 'transformers': '4.57.6'}
    try:
        library_versions = {name: version(name) for name in required_versions}
    except PackageNotFoundError:
        raise ValueError('先にINSTALL=Trueで必要なライブラリを導入してください。') from None
    if any(library_versions[name].split('+', 1)[0] != expected for name, expected in required_versions.items()):
        raise ValueError('確認済みのライブラリ版と異なります。INSTALL=Trueで導入し、ランタイムを再起動してください。')
    import torch, transformers
    library_versions = {'torch': torch.__version__, 'transformers': transformers.__version__}
    if any(library_versions[name].split('+', 1)[0] != expected for name, expected in required_versions.items()):
        raise ValueError('導入前のライブラリがメモリに残っています。ランタイムを再起動してください。')
    print('ライブラリ:', library_versions)
    tokenizer, model = load_model(DEVICE, allow_download=ALLOW_DOWNLOAD)
    print('device:', model.device, 'dtype:', model.dtype)
    print('実際のパラメータ数:', sum(p.numel() for p in model.parameters()))

## 3. 文をトークンに分ける

「京都の名物は」は仕組みを見る人工例です。日本語の文字とトークンが1対1かを確認します。1トークンだけ復号すると文字の一部分になる場合もあります。
**確かめること**：「京都」を別の地名に変えると、文字数とトークン数は同じように変わりますか。英単語に変えた場合も比べられます。

IDと、各トークンを文字へ戻した表示を並べます。一文字が複数トークンに分かれると、単独の表示に`�`が出る場合があります。全IDをつないで元の文へ戻せることも確認してください。


In [ ]:
OBSERVE = False
TEXT = '京都の名物は'
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    ids = None
    observed_text = None
    ids = tokenizer.encode(TEXT, add_special_tokens=False)
    if not 0 < len(ids) <= 64: raise ValueError('観察は1〜64トークンの短文にしてください。')
    observed_text = TEXT
    print('トークン数:', len(ids))
    for token_id in ids:
        print(token_id, repr(tokenizer.decode([token_id])))
    print('元の文へ戻す:', tokenizer.decode(ids))

## 4. トークンIDを数値の列へ変換する
各行が1トークン。先頭4要素から意味を読み取れるとは限りません。文脈処理前の埋込みです。
`torch.inference_mode()`は、重みを学習し直さず計算するときの指定です。`model.device`は、選んだCPUまたはGPUを表します。[PyTorchの説明](https://docs.pytorch.org/docs/2.8/generated/torch.autograd.grad_mode.inference_mode.html)

**確かめること**：文の長さを変えた時、行数と列数のどちらが変わりますか。列数は、このモデルで1トークンを表す数値の個数です。


In [ ]:
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    if globals().get('ids') is None:
        raise ValueError('先にトークン分割のセルを実行してください。')
    if globals().get('observed_text') != TEXT:
        raise ValueError('文字列が変わっています。先にトークン分割をやり直してください。')
    with torch.inference_mode():
        vectors = model.get_input_embeddings()(torch.tensor(ids, device=model.device))
    print('形（トークン数, 次元）:', tuple(vectors.shape))
    print(vectors[:, :4].float().cpu())

## 5. 次のトークンの候補確率を見る
モデルはまず候補ごとのスコア`logits`を出します。これはまだ確率ではなく、負の値や1より大きい値も取れます。`softmax`で語彙全体の合計が1になる確率へ直し、上位5個を表示します。表示するのは`TEXT`の続きの予測で、事実が正しい確率ではありません。

**確かめること**：上位5候補の確率の合計が1より小さいのはなぜでしょう。[Vaswani et al. (2017), §3.4](https://arxiv.org/abs/1706.03762)


In [ ]:
if OBSERVE:
    if globals().get('model') is None or globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    if globals().get('ids') is None or globals().get('observed_text') != TEXT:
        raise ValueError('先に、現在の文字列をトークンに分けるセルを実行してください。')
    token_ids = torch.tensor([ids], device=model.device)
    with torch.inference_mode():
        logits = model(input_ids=token_ids).logits[0, -1].float()
    probabilities = torch.softmax(logits, dim=-1)
    values, candidates = torch.topk(probabilities, 5)
    print('全候補の確率の合計（丸め誤差を含む）:', float(probabilities.sum().cpu()))
    print('表示する上位5候補の合計:', float(values.sum().cpu()))
    for value, token_id in zip(values.cpu(), candidates.cpu()):
        print(repr(tokenizer.decode([int(token_id)])), float(value))

## 6. 自分の質問と入力範囲を用意する

問いは空欄から書きます。下の`source_excerpt`を加える際は資料の利用条件を確認。入力は2,048トークンまで、超えた場合は停止します。Methodsは必要な段落を自分で選びます。
会話テンプレートは全文を表示します。モデル固有の制御記号を含み、非公開の思考過程を取得する機能ではありません。
論文の活動を指定した場合は、生成前に採用・入力条件を検査します。朝のAIなしの段階には切り替えられません。


In [ ]:
PROMPT = ''
source_excerpt = ''
source_location = ''
activity_id = None  # 論文演習では'METHODS'等を生成前に指定。観察だけならNone
PREVIEW = False
previewed_prompt = None
if PREVIEW:
    if globals().get('tokenizer') is None:
        raise ValueError('先にLOAD_MODEL=Trueでモデルを読み込んでください。')
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    visible_template, encoded = encode_prompt(tokenizer, full_prompt)
    print(visible_template)
    print('入力トークン数:', encoded.input_ids.shape[1])
    previewed_prompt = full_prompt

## 7. 短い回答を1回生成する

新規会話、seed 42、temperature 0.7、top_p 0.8、top_k 20、出力96トークン。関数内の`model.generate`にこれらを渡します。出力上限で止まった回答は未完了と表示し、そのまま残します。同じseedでも装置間の完全一致は保証しません。
5の観察は`TEXT`の続きを予測する計算でした。ここでは`PROMPT`を会話テンプレートへ入れ、候補の確率を調整して抽選しながら回答を生成します。観察と生成では入力と選び方が異なります。

**設定の読み方**：`temperature`は候補間の確率の差、`top_k`・`top_p`は候補の絞り方、`seed`は抽選の初期状態を指定します。これらは固定し、まず質問を一つ変えて比べます。[用語集の短い説明](../docs/glossary.md#notebookの生成設定を読む)を参照してください。


In [ ]:
RUN = False
if RUN:
    result = None
    if source_excerpt.strip() and not source_location.strip():
        raise ValueError('抜粋の元ページ・節・図を記入してください。')
    if not PROMPT.strip(): raise ValueError('質問を記入してください。')
    full_prompt = source_excerpt + '\n\n' + PROMPT if source_excerpt else PROMPT
    if globals().get('previewed_prompt') != full_prompt:
        raise ValueError('質問・抜粋を確認するため、プレビューを再実行してください。')
    reading_context = None
    if activity_id is not None:
        reading_context = material_context(course, manifest, activity_id)
        if reading_context['phase'] == 'R0':
            raise ValueError('AIなしの読解は00で記録してください。この段階では生成しません。')
    result = generate(tokenizer, model, full_prompt, max_new_tokens=96, seed=42)
    result['library_versions'] = dict(library_versions)
    result['source_location'] = source_location
    result['reading_context'] = reading_context
    print(result['response_raw'])
    print('終了状態:', result['status'], '秒:', result['elapsed_seconds'])

## 8. 自分の説明を記入し、保存する

`None`は前の記入を保持します。文字列を入れた欄だけ更新し、`SAVE=True`で新しいファイルへ保存します。
生成前に活動を指定していれば、JSONに加えてJSONLとMarkdownも保存します。観察だけならJSONのみです。後から活動を変更しても、生成済みの結果を別の論文に付け替えません。
保存したファイルはFiles欄からダウンロードしてください。


In [ ]:
annotations = {
    'student_explanation': None,  # 自分の説明
    'evidence_location': None,    # 根拠のページ・図・本文
    'unresolved_point': None,     # まだ分からないこと
}
SAVE = False
if SAVE:
    result, paths = save_result(globals().get('result'), annotations, ROOT)
    print('保存:', [str(p.relative_to(ROOT)) for p in paths])